# autofollowdown — full demo

A unified, simple toolkit for compressing AI models — **quantization, pruning, knowledge distillation** — with *real* (non-mock) benchmarks and a one-command flow.

This notebook shows everything `autofollowdown` can do, and explains **which benchmarks** it uses so you can trust the numbers.

Repo: https://github.com/peetwan/autofollowdown

## Setup

On Colab or a fresh machine, install first (uncomment the pip line).

In [1]:
# On Colab / a fresh machine, install first (uncomment):
# !pip install "git+https://github.com/peetwan/autofollowdown#egg=autofollowdown[examples]"
try:
    import autofollowdown as afd
except ImportError:
    # running from a clone (notebooks/ is inside the repo): add the repo root
    import os, sys
    sys.path.insert(0, os.path.abspath('..'))
    import autofollowdown as afd
import warnings; warnings.filterwarnings('ignore')
try:
    import transformers; transformers.logging.set_verbosity_error()
except Exception:
    pass
print('autofollowdown', afd.__version__)

autofollowdown 0.1.0


## 1. What's inside — backends + benchmark catalog

`autofollowdown` always works with its built-in (native) engine, and can route to specialized libraries when they're installed.

In [2]:
from autofollowdown import all_backends
for b in all_backends():
    status = 'installed' if b.is_available() else f'optional  ({b.install_hint})'
    print(f'{b.name:40} {status}')

autofollowdown (native)                  installed
Microsoft NNI                            optional  (pip install nni)
llm-compressor (vLLM)                    optional  (pip install llmcompressor)
NVIDIA TensorRT Model Optimizer          optional  (pip install nvidia-modelopt)


## 2. The three core techniques — real before/after

`autofollowdown` does the three standard compression techniques. We train **one** small CNN on the real scikit-learn `digits` dataset (1,797 real 8x8 digits, no download) and apply each technique to it, measuring the real effect on accuracy, size, and sparsity.

In [3]:
import copy, torch, torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from autofollowdown import (ModelCompressor, count_parameters,
                            evaluate_accuracy, model_disk_size_mb)

torch.manual_seed(0)
d = load_digits()
X = torch.tensor(d.images.astype('float32')/16.0).unsqueeze(1)
y = torch.tensor(d.target, dtype=torch.long)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)
test_loader  = DataLoader(TensorDataset(Xte, yte), batch_size=64)

class CNN(nn.Module):
    def __init__(self, w=32):
        super().__init__()
        self.f = nn.Sequential(nn.Conv2d(1,w,3,padding=1), nn.ReLU(),
                               nn.Conv2d(w,w,3,padding=1), nn.ReLU())
        self.c = nn.Sequential(nn.Flatten(), nn.Linear(w*8*8,128), nn.ReLU(),
                               nn.Linear(128,10))
    def forward(self, x): return self.c(self.f(x))

def train(model, epochs=6):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        for xb, yb in train_loader:
            opt.zero_grad(); lossf(model(xb), yb).backward(); opt.step()
    return model.eval()

baseline = train(CNN())
base_acc  = evaluate_accuracy(baseline, test_loader)
base_size = model_disk_size_mb(baseline)
print(f'trained baseline: accuracy {base_acc:.1%}, size {base_size:.3f} MB')

trained baseline: accuracy 95.6%, size 1.048 MB


### 2a. Quantization — INT8 dynamic
Lower the numeric precision of the weights to 8-bit integers. Smaller model, accuracy preserved.

In [4]:
qmodel = ModelCompressor(copy.deepcopy(baseline)).quantize(method='int8', approach='dynamic').model
print(f'size     : {base_size:.3f} MB  ->  {model_disk_size_mb(qmodel):.3f} MB'
      f'  ({base_size/model_disk_size_mb(qmodel):.2f}x smaller)')
print(f'accuracy : {base_acc:.1%}  ->  {evaluate_accuracy(qmodel, test_loader):.1%}')

size     : 1.048 MB  ->  0.295 MB  (3.55x smaller)
accuracy : 95.6%  ->  95.6%


[W606 22:04:29.664868000 qlinear_dynamic.cpp:252] Warning: Currently, qnnpack incorrectly ignores reduce_range when it is set to true; this may change in a future release. (function operator())


### 2b. Pruning — 50% unstructured (magnitude)
Remove the smallest-magnitude weights and make the zeros permanent. Half the weights become exactly zero.

In [5]:
pmodel = copy.deepcopy(baseline)
_, _, sp0 = count_parameters(pmodel)
ModelCompressor(pmodel).prune(sparsity=0.5, method='unstructured')
_, _, sp1 = count_parameters(pmodel)
print(f'sparsity : {sp0:.0%}  ->  {sp1:.0%}  (half the weights are now exactly 0)')
print(f'accuracy : {base_acc:.1%}  ->  {evaluate_accuracy(pmodel, test_loader):.1%}')

sparsity : 0%  ->  50%  (half the weights are now exactly 0)
accuracy : 95.6%  ->  94.4%


### 2c. Distillation — train a 1/4-width student to mimic the teacher
A real knowledge-distillation training loop (KL on softened logits + cross-entropy on labels). The student is 4x narrower but learns from the teacher.

In [6]:
student = CNN(w=8)   # 1/4 the width of the teacher
ModelCompressor(student).distill(teacher_model=baseline, train_loader=train_loader,
                                 epochs=6, alpha=0.7)
stu_acc, stu_size = evaluate_accuracy(student, test_loader), model_disk_size_mb(student)
print(f'teacher : accuracy {base_acc:.1%}, size {base_size:.3f} MB')
print(f'student : accuracy {stu_acc:.1%}, size {stu_size:.3f} MB'
      f'  ({base_size/stu_size:.2f}x smaller)')

teacher : accuracy 95.6%, size 1.048 MB
student : accuracy 90.4%, size 0.264 MB  (3.97x smaller)


## 3. ⭐ One command: compress every way, benchmark, and pick

`compress_and_benchmark` applies all methods to the trained baseline above, benchmarks them on real data, and recommends the best size/quality trade-off — in one call.

In [7]:
from autofollowdown import compress_and_benchmark
from IPython.display import Markdown, display

study = compress_and_benchmark(baseline, eval_loader=test_loader)
display(Markdown(study.to_markdown()))
print('Recommended:', study.recommended)

| Model | Size (MB) | Params | Sparsity | Latency (ms) | Acc | Fidelity | Size× | Speed× | ΔAcc |
|---|---|---|---|---|---|---|---|---|---|
| baseline | 1.048 | 273,130 | 0.0% | 0.03 | 95.56% | 100.00% | — | — | — |
| int8 dynamic | 0.295 | 9,568 | 0.0% | 0.37 | 95.56% | 99.56% | 3.55× | 0.08× | +0.00% |
| prune 50% | 1.048 | 273,130 | 50.0% | 0.03 | 94.44% | 98.22% | 1.00× | 1.01× | -1.11% |
| prune+quantize | 0.295 | 9,568 | 17.8% | 0.38 | 94.44% | 98.22% | 3.55× | 0.08× | -1.11% |

Recommended: int8 dynamic


Pick a variant by name (or take the recommendation) and export it:

In [8]:
best_model = study.best()                 # the recommended model
# or: chosen = study.pick('int8 dynamic')
path = study.export(study.recommended, 'best_model.pt')
print('saved recommended variant ->', path)

saved recommended variant -> best_model.pt


## 4. Auto-picker — the best *library* for your model

`autofollowdown` profiles a model and ranks compression backends (native always on; NNI / llm-compressor / NVIDIA ModelOpt optional), telling you what's ideal and what's runnable right now.

In [9]:
from autofollowdown import explain
print(explain(baseline))   # our trained CNN

Model profile: ModelProfile(family=cnn, params=273,130, conv=True, transformer=False, hf=False, cuda=False)

Ranked compression backends:
 → 1. [0.60] autofollowdown (native): unstructured-0.5 + int8-dynamic (prune+quantize) — runnable
      Global magnitude pruning then portable INT8 — no GPU needed.
   2. [0.90] Microsoft NNI: L1FilterPruner + ModelSpeedup (structured-prune) — not installed
      Channel/filter pruning that ModelSpeedup turns into a genuinely smaller, faster model (real FLOP reduction).
      install: pip install nni
   3. [0.60] NVIDIA TensorRT Model Optimizer: INT8 PTQ (ptq) — not installed
      Calibrated PTQ (SmoothQuant/AWQ/NVFP4) via mtq.quantize, exportable to TensorRT. Requires an NVIDIA GPU.
      install: pip install nvidia-modelopt

Auto-pick (runnable now): autofollowdown (native)


## 5. Which benchmarks do we use?

Honest evaluation matters. Here is exactly what `autofollowdown` measures.

**Always measured (any model):** on-disk size (MB), parameter count, true sparsity, inference latency (p50), throughput, and — when you pass eval data — top-1 accuracy and output fidelity (agreement with the original model).

**Vision example:** the scikit-learn `digits` dataset (1,797 real 8x8 handwritten digits, ships with scikit-learn — no download). Bring your own DataLoader for your task.

**LLMs — the field-standard suite** (from GPTQ, AWQ, SparseGPT, LLMCBench, LeanQuant, NVIDIA MINITRON, Apple LLM-KICK):

| Pillar | Datasets / tasks | Measures |
|---|---|---|
| Perplexity ↓ | `WikiText-2`, `C4`, `PTB` | language-modeling quality |
| Commonsense (0-shot) | `arc_easy`, `arc_challenge`, `hellaswag`, `winogrande`, `piqa`, `boolq`, `openbookqa`, `lambada` | reasoning |
| Knowledge (5-shot) | `mmlu` | broad factual knowledge |
| Advanced | `mmlu_pro` | harder 10-choice reasoning |
| Multilingual | `mmlu_prox` / `mmlu_prox_lite` (29 languages) | cross-lingual reasoning |
| Reasoning | `gsm8k` | math word problems |
| Truthfulness | `truthfulqa` | reliability |

We compute **WikiText-2 perplexity** ourselves; the accuracy tasks run via EleutherAI's `lm-evaluation-harness`, and we generate the exact command for you.

In [10]:
from autofollowdown import STANDARD_LLM_TASKS, mmlu_prox_tasks, lm_eval_command

for group, tasks in STANDARD_LLM_TASKS.items():
    print(f'{group:22} {tasks}')

print()
# MMLU-ProX multilingual task ids (lite = 658 Q/language, fast for compressed models)
print('MMLU-ProX (en, th, zh):', mmlu_prox_tasks(['en','th','zh'], lite=True))

print()
# The exact command to run the full accuracy suite on a (compressed) model:
print(lm_eval_command('./best_model_hf', tasks=STANDARD_LLM_TASKS['commonsense_zeroshot'] + ['mmlu_prox_lite_en'], num_fewshot=0))

perplexity             ['wikitext2', 'c4', 'ptb']
commonsense_zeroshot   ['arc_easy', 'arc_challenge', 'hellaswag', 'winogrande', 'piqa', 'openbookqa', 'boolq', 'lambada_openai']
knowledge              ['mmlu']
advanced_knowledge     ['mmlu_pro']
multilingual           ['mmlu_prox_en', 'mmlu_prox_lite_en']
reasoning              ['gsm8k']
truthfulness           ['truthfulqa_mc2']

MMLU-ProX (en, th, zh): ['mmlu_prox_lite_en', 'mmlu_prox_lite_th', 'mmlu_prox_lite_zh']

lm_eval --model hf --model_args pretrained=./best_model_hf --tasks arc_easy,arc_challenge,hellaswag,winogrande,piqa,openbookqa,boolq,lambada_openai,mmlu_prox_lite_en --device cuda:0 --batch_size auto --num_fewshot 0


## 6. LLM demo — compress a real pretrained model

Now the real thing: load a pretrained Hugging Face causal LM, compress it with INT8 dynamic quantization, and measure the **headline LLM metric — WikiText-2 perplexity** (lower = better) — plus on-disk size, before and after.

We use [`facebook/opt-125m`](https://huggingface.co/facebook/opt-125m): a real pretrained model whose `nn.Linear` layers INT8 dynamic quantization genuinely shrinks (GPT-2 uses `Conv1D` and barely changes under dynamic quant).

> ⚠️ This cell **downloads ~250 MB** the first time. For real WikiText-2 also `pip install datasets` (otherwise a small built-in text is used and the number isn't comparable).

In [11]:
import copy, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from autofollowdown import (ModelCompressor, evaluate_perplexity,
                            load_wikitext2, model_disk_size_mb)

MODEL_ID = 'facebook/opt-125m'
tok = AutoTokenizer.from_pretrained(MODEL_ID)
llm = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32).eval()

text = load_wikitext2('test')[:3000]   # real WikiText-2 if `datasets` is installed

def measure(m):
    return (model_disk_size_mb(m),
            evaluate_perplexity(m, tok, text, stride=256, max_length=512))

base_size, base_ppl = measure(llm)
qllm = ModelCompressor(copy.deepcopy(llm)).quantize(method='int8', approach='dynamic').model
q_size, q_ppl = measure(qllm)

from IPython.display import Markdown, display
display(Markdown(
    '| Model | Size (MB) | Perplexity (lower=better) | Size\u00d7 | \u0394PPL |\n'
    '|---|---|---|---|---|\n'
    f'| baseline (fp32) | {base_size:.1f} | {base_ppl:.2f} | \u2014 | \u2014 |\n'
    f'| int8 dynamic | {q_size:.1f} | {q_ppl:.2f} | {base_size/q_size:.2f}\u00d7 | {q_ppl-base_ppl:+.2f} |'
))

| Model | Size (MB) | Perplexity (lower=better) | Size× | ΔPPL |
|---|---|---|---|---|
| baseline (fp32) | 477.8 | 30.36 | — | — |
| int8 dynamic | 271.9 | 31.87 | 1.76× | +1.50 |

And the compressed model still generates text:

In [12]:
enc = tok('The future of artificial intelligence is', return_tensors='pt')
with torch.no_grad():
    out = qllm.generate(**enc, max_new_tokens=25, do_sample=False,
                        pad_token_id=tok.eos_token_id)
print(tok.decode(out[0], skip_special_tokens=True))

The future of artificial intelligence is in the hands of the AI.

The future of artificial intelligence is in the hands of the AI.

The


For the full zero-shot + MMLU-ProX accuracy suite, run the CLI / harness command:

In [13]:
# !pip install datasets lm_eval
# !autofollowdown benchmark-llm --model facebook/opt-125m
from autofollowdown import lm_eval_command
print(lm_eval_command(MODEL_ID, num_fewshot=0))

lm_eval --model hf --model_args pretrained=facebook/opt-125m --tasks arc_easy,arc_challenge,hellaswag,winogrande,piqa --device cuda:0 --batch_size auto --num_fewshot 0


## 7. A bigger, more capable model — Qwen3-0.6B (`< 8B`)

`autofollowdown` works on any size of Hugging Face causal LM. Here's a recent, genuinely capable model — [`Qwen/Qwen3-0.6B`](https://huggingface.co/Qwen/Qwen3-0.6B) (596M params) — same one-line compression and the same real WikiText-2 perplexity check.

> ⚠️ Downloads ~1.2 GB and runs on CPU here. Swap `QWEN_ID` for `Qwen/Qwen3-1.7B`, `Qwen/Qwen2.5-3B`, etc. (`< 8B`); for `> 1B` use a GPU (`AutoModelForCausalLM.from_pretrained(QWEN_ID, device_map='auto')`).

In [14]:
import copy, gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from autofollowdown import (ModelCompressor, evaluate_perplexity,
                            load_wikitext2, model_disk_size_mb, explain)

# free the OPT models from the previous section first
for _n in ['llm', 'qllm']:
    if _n in dir(): del globals()[_n]
gc.collect()

QWEN_ID = 'Qwen/Qwen3-0.6B'   # try Qwen3-1.7B / Qwen2.5-3B (use a GPU for >1B)
qtok = AutoTokenizer.from_pretrained(QWEN_ID)
qwen = AutoModelForCausalLM.from_pretrained(QWEN_ID, dtype=torch.float32).eval()
print(f'{QWEN_ID}: {sum(p.numel() for p in qwen.parameters())/1e6:.0f}M params')

text = load_wikitext2('test')[:2000]
def measure(m):
    return model_disk_size_mb(m), evaluate_perplexity(m, qtok, text, stride=512, max_length=1024)

b_size, b_ppl = measure(qwen)
qwen_q = ModelCompressor(copy.deepcopy(qwen)).quantize(method='int8', approach='dynamic').model
q_size, q_ppl = measure(qwen_q)

from IPython.display import Markdown, display
display(Markdown(
    '| Model | Size (MB) | Perplexity (lower=better) | Size\u00d7 | \u0394PPL |\n'
    '|---|---|---|---|---|\n'
    f'| Qwen3-0.6B (fp32) | {b_size:.0f} | {b_ppl:.2f} | \u2014 | \u2014 |\n'
    f'| int8 dynamic | {q_size:.0f} | {q_ppl:.2f} | {b_size/q_size:.2f}\u00d7 | {q_ppl-b_ppl:+.2f} |'
))

Qwen/Qwen3-0.6B: 596M params


| Model | Size (MB) | Perplexity (lower=better) | Size× | ΔPPL |
|---|---|---|---|---|
| Qwen3-0.6B (fp32) | 2274 | 20.37 | — | — |
| int8 dynamic | 1164 | 30.36 | 1.95× | +10.00 |

It still generates coherent text after compression:

In [15]:
enc = qtok('The future of artificial intelligence is', return_tensors='pt')
with torch.no_grad():
    out = qwen_q.generate(**enc, max_new_tokens=20, do_sample=False,
                          pad_token_id=qtok.eos_token_id)
print(qtok.decode(out[0], skip_special_tokens=True))

The future of artificial intelligence is not just about the machines, but about the people who use them. This is the core of the


**Important, honest takeaway.** Notice perplexity rises more here than for the tiny models: naive **dynamic INT8 is a quick, portable baseline, but it costs real quality on capable LLMs**. That's exactly why weight-only, calibrated methods (GPTQ / AWQ) exist — and why `autofollowdown`'s auto-picker recommends `llm-compressor` or `NVIDIA ModelOpt` (not native dynamic) for LLMs:

In [16]:
print(explain(qwen))

Model profile: ModelProfile(family=llm, params=596,049,920, conv=False, transformer=True, hf=True, cuda=False)

Ranked compression backends:
 → 1. [0.40] autofollowdown (native): int8-dynamic (quantize) — runnable
      Portable weight-only INT8 on Linear layers; for 4-bit LLM quality prefer llm-compressor or ModelOpt.
   2. [0.95] llm-compressor (vLLM): GPTQ W4A16 (weight-only-quant) — not installed
      4-bit weight-only PTQ (GPTQ/AWQ) for HF LLMs, deployable in vLLM. GPU strongly recommended.
      install: pip install llmcompressor
   3. [0.90] NVIDIA TensorRT Model Optimizer: INT8 SmoothQuant (ptq) — not installed
      Calibrated PTQ (SmoothQuant/AWQ/NVFP4) via mtq.quantize, exportable to TensorRT. Requires an NVIDIA GPU.
      install: pip install nvidia-modelopt
   4. [0.20] Microsoft NNI: L1FilterPruner + ModelSpeedup (structured-prune) — not installed
      Channel/filter pruning that ModelSpeedup turns into a genuinely smaller, faster model (real FLOP reduction).
      inst

### No network? The same perplexity metric on a tiny in-process model

`perplexity_from_ids` works on any HF causal LM — here a tiny GPT-2 built from a config (no download), useful for offline testing.

In [17]:
import transformers
from autofollowdown import perplexity_from_ids
cfg = transformers.GPT2Config(vocab_size=128, n_positions=64, n_embd=32, n_layer=2, n_head=2)
tiny = transformers.GPT2LMHeadModel(cfg).eval()
ids = torch.randint(0, 128, (1, 160))
print('perplexity (tiny offline model):', round(perplexity_from_ids(tiny, ids, stride=32, max_length=64), 2))

perplexity (tiny offline model): 128.56


## One-liners (after `pip install`)

```bash
autofollowdown auto                      # compress every way, benchmark, pick a winner
autofollowdown info                      # backends + benchmark catalog
autofollowdown benchmark-vision          # offline CNN benchmark
autofollowdown benchmark-llm             # LLM perplexity benchmark
autofollowdown autopick                  # best-library recommendation per model
```

That's the whole toolkit — happy compressing! 🎉